# DeepGuard AI -- GPU Training on Colab

**One-click training pipeline** for the DeepGuard AI deepfake detection model.

- **Runtime**: Go to `Runtime > Change runtime type > T4 GPU`
- **Time**: ~20-30 minutes on T4 GPU
- **Output**: Trained `.keras` model saved to Google Drive

---

## Step 0: Mount Google Drive (saves model safely!)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for our models in Drive
import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/DeepGuard_AI_Models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Models will be saved to: {DRIVE_SAVE_DIR}')

## Step 1: Verify GPU & Clone Repo

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {len(gpus)}')
for gpu in gpus:
    print(f'  {gpu}')
if not gpus:
    print('\n*** No GPU! Go to Runtime > Change runtime type > T4 GPU ***')

In [ ]:
!git clone https://github.com/vishnuwadkar/Deepfake-Detection-Project.git
%cd Deepfake-Detection-Project
!pip install -q mtcnn tqdm scikit-learn

## Step 2: Download Dataset

In [ ]:
from google.colab import files
print('Upload your kaggle.json file:')
print('  Located at: C:\\Users\\vishn\\.kaggle\\kaggle.json')
uploaded = files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API configured!')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p data/raw --unzip

import shutil
from pathlib import Path

src_root = Path('data/raw/real_vs_fake/real-vs-fake')
dst_root = Path('data/processed')

for split in ['train', 'valid', 'test']:
    for cls in ['real', 'fake']:
        src, dst = src_root/split/cls, dst_root/split/cls
        if not dst.exists() and src.exists():
            shutil.copytree(str(src), str(dst))
            print(f'  {split}/{cls}: {len(list(dst.glob("*"))):,}')

print('\nDataset ready!')

## Step 3: Configure for Full GPU Training

In [ ]:
import src.config as cfg

cfg.MAX_TRAIN_SAMPLES = None   # Use ALL 100K images (GPU can handle it)
cfg.BATCH_SIZE = 64            # Optimal for T4 GPU
cfg.EPOCHS_HEAD = 12           # Phase 1
cfg.EPOCHS_FINETUNE = 15       # Phase 2

print('Training config:')
print(f'  Data: ALL 100K images')
print(f'  Batch size: {cfg.BATCH_SIZE}')
print(f'  Phase 1: {cfg.EPOCHS_HEAD} epochs (head only)')
print(f'  Phase 2: {cfg.EPOCHS_FINETUNE} epochs (fine-tune top 50 layers)')

## Step 4: Train (~20-30 min on T4)

This runs both phases. Phase 1 model is auto-saved as backup.

In [ ]:
from src.train import train
train()

## Step 5: Save to Google Drive (IMPORTANT!)

This copies your trained model to Google Drive so you never lose it.

In [ ]:
import shutil

files_to_save = [
    'models/deepfake_detector.keras',
    'models/deepfake_detector_phase1.keras',
    'models/training_history.png',
]

for f in files_to_save:
    if os.path.exists(f):
        dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(f))
        shutil.copy2(f, dest)
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f'  Saved: {os.path.basename(f)} ({size_mb:.1f} MB) -> Drive')

print(f'\nAll models saved to Google Drive: {DRIVE_SAVE_DIR}')
print('Even if runtime disconnects, your models are safe!')

## Step 6: Evaluate

In [ ]:
from src.evaluate import evaluate
evaluate()

# Show report
from IPython.display import Image, display
if os.path.exists('models/evaluation_report.png'):
    display(Image('models/evaluation_report.png', width=900))

# Save report to Drive too
if os.path.exists('models/evaluation_report.png'):
    shutil.copy2('models/evaluation_report.png', DRIVE_SAVE_DIR)
    print('Evaluation report saved to Drive')

## Step 7: Download Model to PC

Download the trained model file. Place it in your project's `models/` folder.

In [ ]:
from google.colab import files

for f in ['models/deepfake_detector.keras', 'models/training_history.png', 'models/evaluation_report.png']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f'Downloading: {os.path.basename(f)} ({size_mb:.1f} MB)')
        files.download(f)

## Step 8 (Optional): Convert to TF.js for Chrome Extension

In [ ]:
!pip install -q tensorflowjs
!python scripts/convert_model.py

# Zip and download TF.js model
import zipfile
from pathlib import Path

model_dir = Path('extension/model')
if model_dir.exists() and any(model_dir.iterdir()):
    with zipfile.ZipFile('tfjs_model.zip', 'w') as z:
        for f in model_dir.iterdir():
            z.write(f, f.name)
    files.download('tfjs_model.zip')
    # Also save to Drive
    shutil.copy2('tfjs_model.zip', DRIVE_SAVE_DIR)
    print('TF.js model saved to Drive and downloading...')
    print('Extract the zip into your extension/model/ folder on your PC.')